Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import json, math, random

Reload vocab + data (self-contained notebook)

In [2]:
with open('../src/vocab.json') as f:
    vocab = json.load(f)
stoi = vocab['stoi']
itos = {int(k): v for k, v in vocab['itos'].items()}
PAD, BOS, EOS = 0, 1, 2
vocab_size = 39

paths, labels = [], []
with open("../data/synth/train/labels.txt") as f:
    for line in f:
        fname, label = line.strip().split("\t")
        paths.append(f"../data/synth/train/{fname}")
        labels.append(label)

transform = T.Compose([
    T.Resize((32, 128)), T.Grayscale(), T.ToTensor(), T.Normalize([0.5], [0.5]),
])

class OCRDataset(Dataset):
    def __init__(self, paths, labels, transform, stoi):
        self.paths, self.labels, self.transform, self.stoi = paths, labels, transform, stoi
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = self.transform(Image.open(self.paths[idx]).convert('RGB'))
        ids = [self.stoi[c] for c in self.labels[idx].lower() if c in self.stoi]
        return img, ids

def collate_fn(batch):
    imgs, label_lists = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(len(l) for l in label_lists) + 1
    tgt_in = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    tgt_out = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    for i, ids in enumerate(label_lists):
        seq = [BOS] + ids
        tgt_in[i, :len(seq)] = torch.tensor(seq)
        out = ids + [EOS]
        tgt_out[i, :len(out)] = torch.tensor(out)
    return imgs, tgt_in, tgt_out

Paste in CNNEncoder and Decoder classes

In [3]:
class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(), nn.MaxPool2d((2, 1)),
        )
    def forward(self, x):
        x = self.conv(x)
        x = x.squeeze(2)
        return x.permute(0, 2, 1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, q_in, kv_in, mask=None):
        B, Tq, _ = q_in.shape
        Tk = kv_in.shape[1]
        Q = self.q_proj(q_in).view(B, Tq, self.h, self.dk).transpose(1, 2)
        K = self.k_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        V = self.v_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = scores.softmax(dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, Tq, -1)
        return self.out_proj(out)

def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(), nn.Linear(ff_dim, d_model))
    def forward(self, x):
        return self.net(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, ff_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    def forward(self, x, memory, self_mask):
        normed = self.norm1(x)
        x = x + self.self_attn(normed, normed, mask=self_mask)
        x = x + self.cross_attn(self.norm2(x), memory, mask=None)
        x = x + self.ffn(self.norm3(x))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024, max_len=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, tgt_in, memory):
        B, T = tgt_in.shape
        x = self.embed(tgt_in) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        mask = causal_mask(T, tgt_in.device)
        for layer in self.layers:
            x = layer(x, memory, mask)
        x = self.norm_out(x)
        return self.fc_out(x)

Combine into one model

In [4]:
class OCRModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024):
        super().__init__()
        self.encoder = CNNEncoder(d_model)
        self.decoder = Decoder(vocab_size, d_model, n_heads, n_layers, ff_dim)

    def forward(self, imgs, tgt_in):
        memory = self.encoder(imgs)
        return self.decoder(tgt_in, memory)

Set up device, model, optimizer

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

model = OCRModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} params")

cuda
4,732,583 params


Loss at step 0 sanity check

In [6]:
ds = OCRDataset(paths, labels, transform, stoi)
loader = DataLoader(ds, batch_size=32, collate_fn=collate_fn, shuffle=True)

imgs, tgt_in, tgt_out = next(iter(loader))
imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)

with torch.no_grad():
    logits = model(imgs, tgt_in)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1), ignore_index=PAD)
print(loss.item())   # expect ~3.66

3.8222899436950684


THE critical check - overfit 32 examples

In [7]:
imgs32, tgt_in32, tgt_out32 = imgs[:32].to(device), tgt_in[:32].to(device), tgt_out[:32].to(device)

model.train()
for step in range(500):
    optimizer.zero_grad()
    logits = model(imgs32, tgt_in32)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out32.reshape(-1), ignore_index=PAD)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(step, loss.item())

0 3.8222899436950684
50 0.7634668350219727
100 0.08535423874855042
150 0.04629003256559372
200 0.02968725934624672
250 0.013780949637293816
300 0.007382480427622795
350 0.0026869275607168674
400 0.0021035457029938698
450 0.001736503909341991


Reset the model fresh

In [8]:
model = OCRModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

LR warmup scheduler

In [9]:
from torch.optim.lr_scheduler import LambdaLR

warmup_steps = 200   
def lr_lambda(step):
    return min((step + 1) / warmup_steps, 1.0)

scheduler = LambdaLR(optimizer, lr_lambda)

Train/val split

In [10]:
n = len(ds)
val_size = int(0.1 * n)
train_ds, val_ds = torch.utils.data.random_split(ds, [n - val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, collate_fn=collate_fn, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn, shuffle=False)

Actual training loop

In [11]:
epochs = 20
model.train()
for epoch in range(epochs):
    total_loss = 0
    for imgs, tgt_in, tgt_out in train_loader:
        imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)

        optimizer.zero_grad()
        logits = model(imgs, tgt_in)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1),
                                ignore_index=PAD, label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"epoch {epoch}: loss {avg_loss:.4f}")

epoch 0: loss 3.8768
epoch 1: loss 3.8173
epoch 2: loss 3.6865
epoch 3: loss 3.4853
epoch 4: loss 3.2857
epoch 5: loss 3.1182
epoch 6: loss 2.9268
epoch 7: loss 2.7290
epoch 8: loss 2.5338
epoch 9: loss 2.3721
epoch 10: loss 2.2098
epoch 11: loss 2.0676
epoch 12: loss 1.9309
epoch 13: loss 1.9587
epoch 14: loss 1.8756
epoch 15: loss 1.7984
epoch 16: loss 1.6215
epoch 17: loss 1.6803
epoch 18: loss 1.5494
epoch 19: loss 1.5546


Add val loss to the loop

In [12]:
def evaluate_loss(model, loader):
    model.eval()
    total = 0
    with torch.no_grad():
        for imgs, tgt_in, tgt_out in loader:
            imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
            logits = model(imgs, tgt_in)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1), ignore_index=PAD)
            total += loss.item()
    model.train()
    return total / len(loader)

In [13]:
model = OCRModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = LambdaLR(optimizer, lr_lambda)

for epoch in range(epochs):
    total_loss = 0
    for imgs, tgt_in, tgt_out in train_loader:
        imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
        optimizer.zero_grad()
        logits = model(imgs, tgt_in)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1),
                                ignore_index=PAD, label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate_loss(model, val_loader)
    print(f"epoch {epoch}: train {train_loss:.4f}  val {val_loss:.4f}")

epoch 0: train 3.8302  val 3.7981
epoch 1: train 3.7729  val 3.6928
epoch 2: train 3.6481  val 3.4914
epoch 3: train 3.4558  val 3.2528
epoch 4: train 3.2511  val 2.9580
epoch 5: train 3.0311  val 2.7908
epoch 6: train 2.9764  val 2.5692
epoch 7: train 2.8405  val 2.4085
epoch 8: train 2.5586  val 2.2574
epoch 9: train 2.4906  val 2.0325
epoch 10: train 2.3224  val 1.8335
epoch 11: train 2.1997  val 1.8100
epoch 12: train 1.9914  val 1.6266
epoch 13: train 1.9518  val 1.4726
epoch 14: train 1.7571  val 1.3429
epoch 15: train 1.6776  val 1.2080
epoch 16: train 1.6507  val 1.2437
epoch 17: train 1.7162  val 1.3100
epoch 18: train 1.6726  val 1.2237
epoch 19: train 1.5496  val 1.1271
